# Challenge 1 — Implement an LRU cache
Question: Design a cache with get(key) and put(key, value) that evicts the least recently used item when its capacity is reached.

Expected approach: Maintain a mapping ordered by recent access. Python's OrderedDict is a straightforward solution.

In [4]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        if capacity <= 0:
            raise ValueError("Capacity must be a positive integer")
        self.capacity = capacity

    def get(self, key: int) -> int:
        if key not in self.cache:
            return None
        else:
            # Move the accessed item to the end to show that it was recently used
            self.cache.move_to_end(key)
            return self.cache[key]

    def put(self, key: int, value: int):
        if key in self.cache:
            # Update the value and move it to the end to show that it was recently used
            self.cache.move_to_end(key)

        self.cache[key] = value

        if len(self.cache) > self.capacity:
            # Remove the first item (the least recently used)
            self.cache.popitem(last=False)

# Example usage:
if __name__ == "__main__":
    cache = LRUCache(2)
    cache.put("A", 100)
    cache.put("B", 200)
    print(cache.get("A"))  # Output: 100
    cache.put("C", 300)    # This will evict key "B"

    print(cache.get("B"))  # Output: None (not found)

100
None


Complexity: Average O(1) for get and put.

Likely follow-ups: What if values can be None? How would you add TTL? Is it thread-safe? How would you make it distributed?

Production answer: For multiple service instances, use Redis if a shared cache is required; an in-memory LRU is local to one process.

# Challenge 2 — Fetch 100 APIs concurrently, but allow only 10 at a time
This is particularly relevant to the job's Python async and enterprise SDK requirements.

In [ ]:
import asyncio
import httpx

async def fetch_all(urls: list[str]):
    semaphore = asyncio.Semaphore(10)  # Limit to 10 concurrent requests

    async with httpx.AsyncClient(timeout=10.0) as client:

        async def fetch(url:str):
            async with semaphore:
                response = await client.get(url)
                response.raise_for_status()  # Raise an error for bad responses
                return response.json()  # Assuming the response is JSON

        tasks = [fetch(url) for url in urls]
        return await asyncio.gather(*tasks)

## What to explain:
- AsyncClient reuses connections.
- Semaphore(10) limits concurrent requests.
- timeout prevents indefinite waits.
- gather() executes I/O operations concurrently and returns results in input order.
- One exception propagates to the caller; other tasks are not automatically cancelled just because gather() raised.
Likely follow-ups: How would you retry HTTP 429 or 503? Preserve partial successes? Cancel outstanding work? Respect a provider-wide quota?

Production improvement: Add bounded retries with jitter, structured logging and a clearly defined partial-failure policy. For very large URL lists, use a fixed-size worker pool rather than create one task per URL.

# Challenge 3 — Implement a retry decorator
Question: Retry transient failures three times with exponential backoff.

In [ ]:
import functools
import random
import time

def retry(max_attempts=3, base_delay=0.5):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except(TimeoutError, ConnectionError):
                    if attempt == max_attempts - 1:
                        raise

                    delay = base_delay * (2 ** attempt)
                    time.sleep(delay + random.uniform(0, 0.1))
        return wrapper
    return decorator

Expected explanation:
"I'd retry only transient failures, use exponential backoff with jitter and limit the number of attempts. I'd respect Retry-After for rate-limited APIs and avoid automatically retrying non-idempotent operations without a safe idempotency mechanism."

Follow-up: Why is time.sleep() unsuitable inside an async function?

It blocks the event loop. An async version should use await asyncio.sleep().

In [ ]:
import httpx

@retry(max_attempts=3, base_delay=0.5)
def fetch_data():
    print("Calling External API")

    response = httpx.get(
        "https://api.example.com/data",
        timeout=5.0
    )

    response.raise_for_status()
    return response.json()

In [ ]:
result = fetch_data()

# Challenge 4 — Implement a rate limiter
Question: Allow a maximum of five requests per user within ten seconds.

Use the sliding-window algorithm.

In [13]:
from collections import defaultdict, deque
from time import monotonic

class RateLimiter:
    def __init__(self, limit=5, window=10):
        self.limit = limit
        self.window = window
        self.requests = defaultdict(deque)

    def allow(self, user_id):
        now = monotonic()
        q = self.requests[user_id]

        while q and q[0] <= now - self.window:
            q.popleft()

        if len(q) >= self.limit:
            return False

        q.append(now)
        return True

Complexity: Amortised O(1) per request because timestamps are appended and removed once.

Production follow-ups: This implementation is process-local, not thread-safe, and doesn't clean up inactive users. For distributed enforcement, consider an atomic Redis implementation using Lua or another server-side atomic operation.

In [ ]:
# Create a rate limiter
limiter = RateLimiter(limit=5, window=10)

# Simulate 7 requests from the same user
for i in range(7):
    allowed = limiter.allow("user123")
    print(f"Request {i + 1}: {'Allowed' if allowed else 'Blocked'}")

Request 1: Allowed
Request 2: Allowed


In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()
limiter = RateLimiter(limit=5, window=10)

@app.get("/app/data")
def get_data(user_id: str):

    if not limiter.allow(user_id):
        raise HTTPException(
            status_code=429,
            detail="Too many requests"
        )

    return {
        "message": "Request successful",
        "user_id": user_id
    }


In [ ]:

import os
import uuid

import redis.asyncio as redis
from fastapi import FastAPI, HTTPException

app = FastAPI()

client = redis.from_url(
    os.getenv("REDIS_URL", "redis://localhost:6379/0"),
    decode_responses=True,
)

# Redis Lua script: atomic sliding-window check
RATE_LIMIT_SCRIPT = """
local key = KEYS[1]
local limit = tonumber(ARGV[1])
local window_ms = tonumber(ARGV[2])
local request_id = ARGV[3]

-- Use the Redis server clock
local t = redis.call('TIME')
local now = t[1] * 1000 + math.floor(t[2] / 1000)

-- Remove requests outside the sliding window
redis.call('ZREMRANGEBYSCORE', key, '-inf', now - window_ms)

-- Count accepted requests still in the window
local count = redis.call('ZCARD', key)

if count >= limit then
    return 0
end

-- Record the accepted request
redis.call('ZADD', key, now, request_id)

-- Automatically delete inactive rate-limit keys
redis.call('PEXPIRE', key, window_ms)

return 1
"""


async def allow_request(user_id: str) -> bool:
    key = f"rate_limit:user:{user_id}"

    allowed = await client.eval(
        RATE_LIMIT_SCRIPT,
        1,
        key,
        5,       # Maximum 5 requests
        10_000,  # Rolling 10-second window
        uuid.uuid4().hex,
    )

    return bool(allowed)


@app.get("/api/data")
async def get_data(user_id: str):
    if not await allow_request(user_id):
        raise HTTPException(
            status_code=429,
            detail="Too many requests. Try again later.",
        )

    return {
        "message": "Request allowed",
        "user_id": user_id,
    }


# Challenge 5 — Process a 20 GB CSV file without loading it into memory

In [3]:
import csv

def read_records(path: str):
    with open(path, newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)

def process_records(path: str):
    for record in read_records(path):
        if record.get("status") == "active":
            yield record


In [4]:
for record in process_records("../em.csv"):
    print(record)

# Challenge 6 — Build a document chunker for RAG
Question: Split a document into overlapping chunks.

In [ ]:
def chunk_tokens(
    tokens: list[str],
    chunk_size: int=500,
    overlap: int=50
) -> list[list[str]]:
    if not 0 <= overlap < chunk_size:
        raise ValueError("Invalid chunk configuration")

    step = chunk_size - overlap

    return [
        tokens[i:1+chunk_size] for i in range(0, len(tokens), step)
    ]

# Challenge 7 — Implement cosine similarity

In [ ]:
import math

def consine_similarity(a,b):
    if len(a) != len(b):
        raise ValueError("Dimension mismatch")

    dot = sum(x*y for x,y in zip(a,b))
    norm_a = math.sqrt(sum(x*x for x in a))
    norm_b = math.sqrt(sum(y*y for y in b))

    if norm_a == 0 or norm_b == 0:
        raise ValueError("Zero vector")

    return dot / (norm_a * norm_b)

# Flatten nested JSON

In [ ]:
data = {
  "user": {
    "name": "John",
    "address": {
      "city": "London",
      "postcode": "SW1"
    }
  },
  "active": true
}

convert to {
  "user.name": "John",
  "user.address.city": "London",
  "user.address.postcode": "SW1",
  "active": true
}

In [9]:
def flatten_jason(data, parent_key="", sep="."):
    result = {}

    for key, value in data.items():
        new_key = (
            f"{parent_key}{sep}{key}" if parent_key else str(key)
        )

        if isinstance(value, dict):
            result.update(
                flatten_jason(value, new_key, sep)
            )

        elif isinstance(value, list):
            for index, item in enumerate(value):
                list_key = f"{new_key}{sep}{index}"

                if isinstance(item, dict):
                    result.update(
                        flatten_jason(item, list_key, sep)
                    )
                else:
                    result[list_key] = item

        else:
            result[new_key] = value

    return result

In [10]:
data = {
    "user": {
        "name": "John",
        "address": {
            "city": "London",
            "postcode": "SW1"
        }
    },
    "skills": ["Python", "Django"]
}

flatten_jason(data)

{'user.name': 'John',
 'user.address.city': 'London',
 'user.address.postcode': 'SW1',
 'skills.0': 'Python',
 'skills.1': 'Django'}

# give example for this:  Merge overlapping intervals 
Sorting, complexity
Merge Overlapping Intervals is a common Python coding interview question that tests your understanding of sorting, arrays, iteration and time complexity.

The objective is to combine intervals that overlap into a single interval.

In [ ]:
intervals = [[1, 3], [2, 6], [8, 10], [9, 12], [15, 18]]

#Expected Output
#[[1, 6], [8, 12], [15, 18]]

In [11]:
def merge_intervals(intervals):
    if not intervals:
        return []

    # Step 1: Sort intervals by start time
    intervals = sorted(intervals, key=lambda x: x[0])

    # Step 2: Initialise result with first interval
    merged = [intervals[0].copy()]

    # Step 3: Process remaining intervals
    for start, end in intervals[1:]:

        last = merged[-1]

        # Step 4: Check whether intervals overlap
        if start <= last[1]:
            last[1] = max(last[1], end)

        else:
            # No overlap: add a new interval
            merged.append([start, end])

    return merged

In [12]:
intervals = [
    [1, 3],
    [2, 6],
    [8, 10],
    [9, 12],
    [15, 18]
]

result = merge_intervals(intervals)

print(result)

[[1, 6], [8, 12], [15, 18]]


# Deduplicate records while preserving order	Hashing, generators

In [ ]:
def deduplicate(records):
    seen = set()

    for record in records:
        if record not in seen:
            seen.add(record)
            yield record

In [ ]:
records = [10, 20, 10, 30, 20, 40, 30, 50]

result = list(deduplicate(records))

print(result)

# Implementing a TTL (Time-To-Live) Cache is a common Python coding interview question that tests your understanding of dictionaries, hashing, time-based expiration, caching semantics and thread safety.

A TTL cache stores a value temporarily. Once its configured lifetime expires, the value is considered invalid and must be fetched or calculated again.

For example, imagine caching vehicle information from an external API for 10 seconds.
- First request: Fetch the vehicle information and cache it.
- Second request within 10 seconds: Return the cached value.
- Request after 10 seconds: Cache has expired, so fetch fresh information.

In [ ]:
from time import monotonic


class TTLCache:
    def __init__(self, ttl=10):
        if ttl <= 0:
            raise ValueError("TTL must be positive")

        self.ttl = ttl
        self.cache = {}

    def set(self, key, value):
        expires_at = monotonic() + self.ttl
        self.cache[key] = (value, expires_at)

    def get(self, key):
        entry = self.cache.get(key)

        if entry is None:
            return None

        value, expires_at = entry

        if monotonic() >= expires_at:
            del self.cache[key]
            return None

        return value

In [ ]:
import time

cache = TTLCache(ttl=5)

cache.set("user:101", {"name": "John"})

print(cache.get("user:101"))
# {'name': 'John'}

time.sleep(3)

print(cache.get("user:101"))
# {'name': 'John'}

time.sleep(3)

print(cache.get("user:101"))
# None (expired)